In [ ]:
%load_ext autoreload
%autoreload 2

## Investigating the Unembed Weight Matrix

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from mpl_toolkits.axes_grid1 import make_axes_locatable

from tsfm_lens.chronos.circuitlens import CircuitLensChronos
from tsfm_lens.utils import (
    apply_custom_style,
    make_ensemble_from_arrow_dir,
    set_seed,
)

In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../figs", "decoder_vis")
os.makedirs(plot_save_dir, exist_ok=True)


In [ ]:
pipeline = CircuitLensChronos.from_pretrained("amazon/chronos-t5-base", device_map=device, torch_dtype=torch.bfloat16)
num_layers = pipeline.model.model.config.num_decoder_layers
num_heads = pipeline.model.model.config.num_heads

print(f"num_layers: {num_layers}, num_heads: {num_heads}")

In [ ]:
split_name = "base40"
split_dir = os.path.join(DATA_DIR, split_name)

system_name = "Lorenz"
subsample_interval = 1

ensemble = make_ensemble_from_arrow_dir(split_dir, dyst_names_lst=[system_name], num_samples=1)

In [ ]:
context_length = 512
context_start_time = 1000  # 1280
context_end_time = context_start_time + context_length

# Prepare input series
sample_idx = 0
selected_dim = 0
# get sample sample_idx of Lorenz
dyst_coords = torch.tensor(ensemble["Lorenz"][sample_idx, selected_dim, :]).unsqueeze(0)
print(dyst_coords.shape)
dyst_coords = dyst_coords[:, ::subsample_interval]
context = dyst_coords[:, context_start_time:context_end_time]

print(context.shape)

In [ ]:
prediction_length = 512

future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]
print(f"future_vals shape: {future_vals.shape}")

In [ ]:
rseed = 123
set_seed(rseed)

### W_unembed (lm head)

In [ ]:
lm_head_weight = pipeline.model.model.lm_head.weight.detach().cpu().float().numpy()
print(f"lm_head_weight shape: {lm_head_weight.shape}")

In [ ]:
plt.figure(figsize=(10, 6))
vabs = np.nanmax(np.abs(lm_head_weight)) if np.any(np.isfinite(lm_head_weight)) else 0
im = plt.imshow(
    lm_head_weight,
    aspect="auto",
    cmap="RdBu",
    vmin=-vabs,
    vmax=vabs,
)
plt.colorbar(im, label="Weight Value", pad=0.01)
plt.title("LM Head Weight Matrix", fontweight="bold")
plt.xlabel("Hidden Dimension", fontweight="bold")
plt.ylabel("Token ID", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(plot_save_dir, "lm_head_weight_matrix.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
gramian_matrix = lm_head_weight @ lm_head_weight.T

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
# Set colorbar limits to [-500, 500], clipping values outside this range
vmax = 300
vmin = -300
clipped_gramian_matrix = np.clip(gramian_matrix, vmin, vmax)
im = ax.imshow(
    clipped_gramian_matrix,  # clip values for display
    aspect="equal",
    cmap="RdBu",
    vmin=vmin,
    vmax=vmax,
)
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im, cax=cax)
ax.set_title("Gramian of LM Head Weights", fontweight="bold")
ax.invert_yaxis()  # Flip the y axis ticks
plt.tight_layout()
plt.savefig(os.path.join(plot_save_dir, "lm_head_gramian.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
gramian_matrix = lm_head_weight.T @ lm_head_weight

# Use eigh for more numerical stability and ensure Hermitian input
# eigenvalues = np.linalg.eigvalsh((gramian_matrix + gramian_matrix.T) / 2)
eigenvalues = np.linalg.eigvalsh(gramian_matrix)

eigenvalues_sorted = np.sort(eigenvalues)[::-1]

# check if there are any negative eigenvalues
if np.any(eigenvalues < 0):
    num_negative_eigenvalues = np.sum(eigenvalues < 0)
    print(f"There are {num_negative_eigenvalues} negative eigenvalues")
else:
    print("There are no negative eigenvalues")

plt.figure(figsize=(5, 4))
plt.plot(eigenvalues_sorted, marker="o", markersize=3, alpha=0.8)
plt.title("Ordered Gramian Eigenvalues", fontweight="bold")
plt.xlabel("Index", fontweight="bold")
plt.ylabel("Eigenvalue", fontweight="bold")
# plt.grid(True)
plt.yscale("log")
plt.xscale("log")
plt.tight_layout()
plt.savefig(
    os.path.join(plot_save_dir, "lm_head_gramian_ordered_eigenvalues.pdf"),
    bbox_inches="tight",
)
plt.show()


In [ ]:
plt.figure(figsize=(5, 4))
bins = np.logspace(np.log10(np.min(eigenvalues)), np.log10(np.max(eigenvalues)), 80)
plt.hist(eigenvalues, bins=bins, histtype="stepfilled", alpha=0.8)
plt.xlabel("Eigenvalue", fontweight="bold")
plt.ylabel("Count", fontweight="bold")
plt.yscale("log")
plt.xscale("log")
plt.tight_layout()
plt.savefig(
    os.path.join(plot_save_dir, "lm_head_gramian_eigenvalues_histogram.pdf"),
    bbox_inches="tight",
)
plt.show()


In [ ]:
# Compute Gramian matrix
gramian_matrix = lm_head_weight @ lm_head_weight.T

# Compute PCA (eigen-decomposition)
eigvals, eigvecs = np.linalg.eigh(gramian_matrix)
# Sort eigenvalues and eigenvectors in descending order
idx = np.argsort(eigvals)[::-1]
eigvals_sorted = eigvals[idx]
eigvecs_sorted = eigvecs[:, idx]

# Plot the top principal components (eigenvectors)
num_components_to_plot = 6  # You can change this number as needed
plt.figure(figsize=(10, 6))
for i in range(num_components_to_plot):
    plt.plot(eigvecs_sorted[:, i], label=f"PC {i + 1}")
plt.title("Top Principal Components of Gramian Matrix", fontweight="bold")
plt.xlabel("Token ID", fontweight="bold")
plt.ylabel("Component Value", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.savefig(
    os.path.join(plot_save_dir, "lm_head_gramian_eigenvectors.pdf"),
    bbox_inches="tight",
)
plt.show()

In [ ]:
eigvecs_sorted[:, 0].shape

In [ ]:
# Corner plot of top 5 principal components (eigenvectors)
num_pcs = 6
fig, axes = plt.subplots(num_pcs, num_pcs, figsize=(3 * num_pcs, 3 * num_pcs))

for i in range(num_pcs):
    for j in range(num_pcs):
        ax = axes[i, j]
        if i == j:
            # Diagonal: 1D histogram of the i-th principal component
            ax.hist(eigvecs_sorted[:, i], bins=50, color="C0", alpha=0.7)
            # ax.set_ylabel(f"PC {i+1}", fontweight="bold")
            ax.set_title(f"PC {i + 1}", fontweight="bold")
            # ax.set_yticks([])
        elif i > j:
            # Lower triangle: 2D histogram of (PC_j, PC_i)
            x = eigvecs_sorted[:, j]
            y = eigvecs_sorted[:, i]
            h = ax.hist2d(x, y, bins=50, cmap="Blues")
            # No axis labels for off-diagonal plots
        else:
            # Upper triangle: turn off axis
            ax.axis("off")

# Adjust layout
plt.subplots_adjust(top=0.92)
plt.tight_layout()
plt.savefig(
    os.path.join(plot_save_dir, "lm_head_gramian_eigenspaces.pdf"),
    bbox_inches="tight",
)
plt.show()


In [ ]:
print("lm_head_weight shape:", lm_head_weight.shape)

# Take SVD of lm_head_weight
U, S, Vh = np.linalg.svd(lm_head_weight, full_matrices=False)
print("U shape:", U.shape)
print("S shape:", S.shape)
print("Vh shape:", Vh.shape)


In [ ]:
explained_energy = np.cumsum(S[:num_pcs] ** 2) / np.sum(S**2)

plt.figure(figsize=(4, 4))
plt.plot(np.arange(1, num_pcs + 1), explained_energy, marker="o")
plt.xlabel("Top-k components", fontweight="bold")
plt.title("Variance Explained", fontweight="bold")
plt.tight_layout()
plt.savefig(
    os.path.join(plot_save_dir, "lm_head_gramian_variance_explained.pdf"),
    bbox_inches="tight",
)
plt.show()

In [ ]:
context.shape